# Data obfuscation techniques on SWIFT transaction data

Generate synthetic SWIFT records, then apply each obfuscation technique from the obfuscation study: field-level masking, tokenization, fuzzy range blurring, noisy aggregation, metadata sanitization and k-anonymity. Every figure becomes a workbench artifact.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # repo root (kernel cwd)

from examples.obfuscation.swift_data import generate_swift
from examples.obfuscation import obfuscate as obf

df = generate_swift(2000, seed=42)
print(df.shape)
df[['sender_iban', 'sender_bic_swift_code', 'sender_institution_name',
    'sender_city', 'transaction_amount_usd']].head(3)

In [ ]:
masked = obf.apply_masking(df, mask=['sender_iban', 'sender_bic_swift_code',
                                        'sender_institution_name', 'sender_city'])
masked[['sender_iban', 'sender_bic_swift_code',
        'sender_institution_name', 'sender_city']].head(3)

In [ ]:
tok = obf.tokenize(df, columns=['sender_iban', 'receiver_iban', 'sender_bic_swift_code'])
print('IBAN -> token:', df['sender_iban'][0], '->', tok['sender_iban'][0])
print('Amount exact:', df['transaction_amount_usd'][0],
      '-> fuzzy:', obf.fuzzy_bucket(df['transaction_amount_usd'][0], width=5000))

anon, risk = obf.k_anonymize(df, ['booking_date', 'sender_city', 'transaction_amount_usd'], k=5)
print(f'Rows still in k<5 quasi-id classes after k-anonymity: {risk:.1%}')

In [ ]:
import matplotlib.pyplot as plt
rows = df.head(6)
x = range(len(rows))
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
axes[0].bar([i-0.2 for i in x], rows['sender_iban'].str.len(), width=0.4, label='original', color='#e05b5b')
axes[0].bar([i+0.2 for i in x], obf.apply_masking(df, mask=['sender_iban']).head(6)['sender_iban'].str.len(),
            width=0.4, label='masked', color='#35c4b6')
axes[0].set_ylabel('IBAN length (chars)'); axes[0].set_title('Masking preserves structure')
axes[0].legend()
axes[1].bar(['exact', 'fuzzy $5K'], [1.0, 1/8], color=['#e05b5b', '#35c4b6'])
axes[1].set_yscale('log'); axes[1].set_ylabel('guessing probability')
axes[1].set_title('Fuzzy ranges crush KBA guessing odds')